In [17]:
from pathlib import Path

import pandas as pd


def data_dir() -> Path:
    """Thư mục `data` cạnh `scripts` (data-platform), không phụ thuộc cwd."""
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        candidate = d / "data"
        if candidate.is_dir() and (candidate / "olist_orders_dataset.csv").exists():
            return candidate
    raise FileNotFoundError("Không tìm thấy data-platform/data (có olist_orders_dataset.csv).")


DATA_DIR = data_dir()
print(DATA_DIR)

D:\H\Projects\data-platform\data


In [18]:
orders      = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
items       = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
customers   = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
products    = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
sellers     = pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv")
payments    = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
reviews     = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")
geo         = pd.read_csv(DATA_DIR / "olist_geolocation_dataset.csv")
translation = pd.read_csv(DATA_DIR / "product_category_name_translation.csv")

In [19]:
from IPython.display import display


def eda_bundle(tables, *, cat_max_unique=30, sample_rows=2):
    """Một lần chạy: overview toàn bộ bảng + null % + describe số + value_counts cột ít giá trị."""
    overview = []
    for name, df in tables.items():
        mem = df.memory_usage(deep=True).sum() / 1e6
        overview.append(
            {
                "table": name,
                "rows": len(df),
                "cols": df.shape[1],
                "memory_mb": round(mem, 2),
                "dup_rows": int(df.duplicated().sum()),
            }
        )
    display(pd.DataFrame(overview).set_index("table"))

    for name, df in tables.items():
        print("\n" + "=" * 60)
        print(f"TABLE: {name}  |  {len(df):,} rows × {df.shape[1]} cols")
        null_pct = (df.isnull().mean() * 100).round(2).sort_values(ascending=False)
        print("\nNull % (cột có null, tối đa 10):")
        nz = null_pct[null_pct > 0].head(10)
        print(nz.to_string() if len(nz) else "  (không có null)")

        num = df.select_dtypes(include="number")
        if not num.empty:
            print("\nNumeric describe:")
            display(num.describe().T)

        low_card = []
        for col in df.columns:
            if df[col].dtype == object or pd.api.types.is_string_dtype(df[col]):
                nu = df[col].nunique(dropna=True)
                if nu <= cat_max_unique:
                    low_card.append(col)
        for col in low_card:
            vc = df[col].value_counts(dropna=False)
            print(f"\n[{col}] nunique={df[col].nunique(dropna=True)} — value_counts:")
            print(vc.to_string())

        print(f"\nSample ({sample_rows} rows):")
        display(df.head(sample_rows))


def olist_sanity(t):
    """Trả lời nhanh các câu checklist Olist (dựa trên tên biến chuẩn notebook)."""
    o, it, pay, rev, g = t["orders"], t["items"], t["payments"], t["reviews"], t["geo"]

    print("=== ORDERS ===")
    print("order_status:", sorted(o["order_status"].dropna().unique().tolist()))
    worst_null = (o.isnull().mean() * 100).idxmax()
    print("Cột null % cao nhất:", worst_null, f"({o[worst_null].isnull().mean()*100:.2f}%)")

    print("\n=== ITEMS ===")
    per_order = it.groupby("order_id").size()
    print("Số dòng tối đa / 1 order_id:", int(per_order.max()))
    print("Cột giá: `price` (giá sản phẩm), `freight_value` (phí ship)")

    print("\n=== PAYMENTS ===")
    print("payment_type:", sorted(pay["payment_type"].dropna().unique().tolist()))
    multi = pay.groupby("order_id").size()
    print("Order có >1 payment:", int((multi > 1).sum()), "/", len(multi))

    print("\n=== REVIEWS ===")
    print("review_score min–max:", int(rev["review_score"].min()), "–", int(rev["review_score"].max()))
    msg = rev["review_comment_message"]
    print("Null review_comment_message:", f"{msg.isnull().mean()*100:.2f}%")

    print("\n=== GEO ===")
    print("rows:", f"{len(g):,}")
    z = g["geolocation_zip_code_prefix"]
    print("Số zip prefix khác nhau:", int(z.nunique()), "→ nhiều dòng / zip (duplicate theo prefix là bình thường).")
    print("Số dòng / 1 zip prefix (median):", float(z.value_counts().median()))
    print("Lý do nhiều rows: cùng mã zip có nhiều cặp lat/lng (khu phố khác nhau).")

In [20]:
TABLES = {
    "orders": orders,
    "items": items,
    "customers": customers,
    "products": products,
    "sellers": sellers,
    "payments": payments,
    "reviews": reviews,
    "geo": geo,
    "translation": translation,
}

# Toàn bộ bảng: bảng tổng quan + chi tiết từng table
eda_bundle(TABLES)

# Các câu checklist Olist (orders/items/payments/reviews/geo)
olist_sanity(TABLES)

,rows,cols,memory_mb,dup_rows
table,,,,
orders,99441,8,55.51,0
items,112650,7,37.74,0
customers,99441,5,27.88,0
products,32951,9,6.60,0
sellers,3095,4,0.62,0
payments,103886,5,17.02,0
reviews,99224,7,41.03,0
geo,1000163,5,135.67,261831
translation,71,2,0.01,0



TABLE: orders  |  99,441 rows × 8 cols

Null % (cột có null, tối đa 10):
order_delivered_customer_date    2.98
order_delivered_carrier_date     1.79
order_approved_at                0.16

[order_status] nunique=8 — value_counts:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2

Sample (2 rows):


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00



TABLE: items  |  112,650 rows × 7 cols

Null % (cột có null, tối đa 10):
  (không có null)

Numeric describe:


,count,mean,std,min,25%,50%,75%,max
order_item_id,112650.0,1.197834,0.705124,1.00,1.00,1.00,1.00,21.00
price,112650.0,120.653739,183.633928,0.85,39.90,74.99,134.90,6735.00
freight_value,112650.0,19.990320,15.806405,0.00,13.08,16.26,21.15,409.68


...


[geolocation_state] nunique=27 — value_counts:
geolocation_state
SP    404268
MG    126336
RJ    121169
RS     61851
PR     57859
SC     38328
BA     36045
GO     20139
ES     16748
PE     16432
DF     12986
MT     12031
CE     11674
PA     10853
MS     10431
MA      7853
PB      5538
RN      5041
PI      4549
AL      4183
TO      3576
SE      3563
RO      3478
AM      2432
AC      1301
AP       853
RR       646

Sample (2 rows):


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP



TABLE: translation  |  71 rows × 2 cols

Null % (cột có null, tối đa 10):
  (không có null)

Sample (2 rows):


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories


=== ORDERS ===
order_status: ['approved', 'canceled', 'created', 'delivered', 'invoiced', 'processing', 'shipped', 'unavailable']
Cột null % cao nhất: order_delivered_customer_date (2.98%)

=== ITEMS ===
Số dòng tối đa / 1 order_id: 21
Cột giá: `price` (giá sản phẩm), `freight_value` (phí ship)

=== PAYMENTS ===
payment_type: ['boleto', 'credit_card', 'debit_card', 'not_defined', 'voucher']
Order có >1 payment: 2961 / 99440

=== REVIEWS ===
review_score min–max: 1 – 5
Null review_comment_message: 58.70%

=== GEO ===
rows: 1,000,163
Số zip prefix khác nhau: 19015 → nhiều dòng / zip (duplicate theo prefix là bình thường).
Số dòng / 1 zip prefix (median): 29.0
Lý do nhiều rows: cùng mã zip có nhiều cặp lat/lng (khu phố khác nhau).


# Tổng hợp EDA — Olist (Brazilian E-Commerce)

## Quy mô

| Bảng | Dòng | Cột |
|------|-----:|----:|
| orders | 99.441 | 8 |
| customers | 99.441 | 5 |
| items | 112.650 | 7 |
| payments | 103.886 | 5 |
| reviews | 99.224 | 7 |
| products | 32.951 | 9 |
| sellers | 3.095 | 4 |
| geo | 1.000.163 | 5 |
| translation | 71 | 2 |

## Orders
- **order_status:** 8 giá trị; chủ yếu `delivered` (~97k), sau đó shipped, canceled, unavailable, invoiced, processing; rất ít `created` / `approved`.
- **Null nhiều nhất:** `order_delivered_customer_date` (~3%), `order_delivered_carrier_date` (~1,8%), `order_approved_at` (~0,16%).
- Cột thời gian trong CSV dạng chuỗi → cần `pd.to_datetime` khi modeling.

## Items
- Một **order_id** có thể có **nhiều dòng** (tối đa ~21; trung bình ~1,14 dòng/đơn).
- **Giá:** `price` (item), `freight_value` (phí vận chuyển).

## Payments
- **payment_type:** boleto, credit_card, debit_card, not_defined, voucher.
- **~2,9k** đơn có **>1** dòng thanh toán.

## Reviews
- **review_score:** 1–5.
- **~58,7%** `review_comment_message` null; title null rất nhiều → bình thường với review chỉ có điểm.

## Geolocation
- **~1M** dòng; **~19k** zip prefix khác nhau → **nhiều lat/lng / một prefix** (median ~29 dòng/prefix). Join geo cần rule (aggregate hoặc chọn 1 dòng).

## Products
- **~1,85%** null `product_category_name` → xử lý khi join `translation`.

## Rủi ro modeling
- Zip đọc từ CSV dạng số có thể **mất số 0 đầu** → chuẩn hóa string 5 ký tự khi join customers/sellers/geo.
- Grain khác nhau: đơn vs item vs payment vs review → star schema cần chọn fact table đúng.

*Bản lưu cùng nội dung: `notes/eda-summary.md`.*

## Bước cleaning (sau EDA)

1. **Datetime:** parse cột thời gian ở `orders`, `items`, `reviews`.
2. **Zip / CEP:** đồng nhất prefix 5 ký tự (zero-pad) cho customers, sellers, geo — tránh lệch join.
3. **products:** giữ null category hoặc gán `unknown` tùy policy staging.
4. **geo (tùy chọn):** bảng `geo_agg` = median lat/lng theo prefix để dim địa lý 1 dòng/prefix.
5. **Output:** lưu `data/clean/*.parquet` (hoặc CSV) để bước BigQuery/dbt dùng lại.

In [23]:
# --- Cleaning: chuẩn hóa type + datetime + zip, ghi parquet ---

CLEAN_DIR = DATA_DIR.parent / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)


def _zip5(series: pd.Series) -> pd.Series:
    """CEP prefix → str 5 ký tự (sửa mất số 0 đầu khi CSV đọc thành int)."""
    s = pd.to_numeric(series, errors="coerce")
    out = pd.Series(pd.NA, index=series.index, dtype="string")
    m = s.notna()
    out.loc[m] = s.loc[m].round().astype(int).astype(str).str.zfill(5)
    return out


def clean_orders(df: pd.DataFrame) -> pd.DataFrame:
    o = df.copy()
    for c in [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ]:
        o[c] = pd.to_datetime(o[c], errors="coerce")
    return o


def clean_items(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy()
    x["shipping_limit_date"] = pd.to_datetime(x["shipping_limit_date"], errors="coerce")
    return x


def clean_customers(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy()
    x["customer_zip_code_prefix"] = _zip5(x["customer_zip_code_prefix"])
    return x


def clean_sellers(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy()
    x["seller_zip_code_prefix"] = _zip5(x["seller_zip_code_prefix"])
    return x


def clean_geo(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy()
    x["geolocation_zip_code_prefix"] = _zip5(x["geolocation_zip_code_prefix"])
    return x


def clean_reviews(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy()
    x["review_creation_date"] = pd.to_datetime(x["review_creation_date"], errors="coerce")
    x["review_answer_timestamp"] = pd.to_datetime(x["review_answer_timestamp"], errors="coerce")
    return x


def geo_by_prefix(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby("geolocation_zip_code_prefix", as_index=False).agg(
        geolocation_lat=("geolocation_lat", "median"),
        geolocation_lng=("geolocation_lng", "median"),
        n_rows=("geolocation_lat", "count"),
    )
    return g


orders_c = clean_orders(orders)
items_c = clean_items(items)
customers_c = clean_customers(customers)
sellers_c = clean_sellers(sellers)
reviews_c = clean_reviews(reviews)
geo_c = clean_geo(geo)
geo_agg = geo_by_prefix(geo_c)
# products, translation: giữ nguyên (chỉ cần join sau); optional fill category
products_c = products.copy()
products_c["product_category_name"] = products_c["product_category_name"].fillna("unknown")
translation_c = translation.copy()

out = {
    "olist_orders_dataset": orders_c,
    "olist_order_items_dataset": items_c,
    "olist_customers_dataset": customers_c,
    "olist_sellers_dataset": sellers_c,
    "olist_order_payments_dataset": payments.copy(),
    "olist_order_reviews_dataset": reviews_c,
    "olist_products_dataset": products_c,
    "olist_geolocation_dataset": geo_c,
    "geolocation_by_zip_prefix": geo_agg,
    "product_category_name_translation": translation_c,
}

for name, d in out.items():
    path = CLEAN_DIR / f"{name}.parquet"
    d.to_parquet(path, index=False)
    print("Wrote", path, f"({len(d):,} rows)")

print("\nDone. Raw CSV giữ tại data/; bản clean tại", CLEAN_DIR)

Wrote D:\H\Projects\data-platform\clean\olist_orders_dataset.parquet (99,441 rows)
Wrote D:\H\Projects\data-platform\clean\olist_order_items_dataset.parquet (112,650 rows)
Wrote D:\H\Projects\data-platform\clean\olist_customers_dataset.parquet (99,441 rows)
Wrote D:\H\Projects\data-platform\clean\olist_sellers_dataset.parquet (3,095 rows)
Wrote D:\H\Projects\data-platform\clean\olist_order_payments_dataset.parquet (103,886 rows)
Wrote D:\H\Projects\data-platform\clean\olist_order_reviews_dataset.parquet (99,224 rows)
Wrote D:\H\Projects\data-platform\clean\olist_products_dataset.parquet (32,951 rows)
Wrote D:\H\Projects\data-platform\clean\olist_geolocation_dataset.parquet (1,000,163 rows)
Wrote D:\H\Projects\data-platform\clean\geolocation_by_zip_prefix.parquet (19,015 rows)
Wrote D:\H\Projects\data-platform\clean\product_category_name_translation.parquet (71 rows)

Done. Raw CSV giữ tại data/; bản clean tại D:\H\Projects\data-platform\clean
